# Pipeline benchmark — reviewed windows vs hand-verified truth

Runs the pipeline **unattended** on the two reviewed windows of the full-half videos, then
scores it against the hand-corrected `*_unified.csv` files (IDF1, ID switches, MOTA) plus
the no-ground-truth proxy metrics. About 3,700 frames in total, roughly 15–25 minutes on a T4.

**Before running**
1. `Runtime → Change runtime type → T4 GPU`.
2. Colab Secrets (🔑, left sidebar): `ROBOFLOW_API_KEY`.
3. In Google Drive, folder `MyDrive/Playbook/benchmark/` containing:
   - `half1.mp4`, `half2.mp4` — the **full, untrimmed** half videos the original CSVs came from
   - `per_frame_tracks_half1_unified.csv`, `per_frame_tracks_half2_unified.csv`

**Running (people or Claude in Chrome):** run cells top to bottom, once each. Cell 6 is the
long one; it skips any half that already finished, so after a disconnect just re-run from
cell 1. If a cell fails, stop and report its last output lines. Results are saved to
`MyDrive/Playbook/benchmark/results_<timestamp>/`.

In [ ]:
# Cell 1 — Verify GPU
import subprocess
g = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                   capture_output=True, text=True)
if g.returncode != 0:
    raise SystemExit('No GPU. Runtime -> Change runtime type -> T4 GPU, then re-run.')
print('GPU:', g.stdout.strip())

In [ ]:
# Cell 2 — Install (GPU build), once. CUDA onnxruntime goes LAST so nothing shadows it.
!apt-get install -qq ffmpeg libglib2.0-0 libsm6 libxext6 libxrender-dev >/dev/null
!pip uninstall -qqy opencv-python opencv-python-headless >/dev/null 2>&1
!pip install -q \
    'numpy>=2.0.0,<2.4.0' opencv-python-headless==4.10.0.84 tqdm 'requests>=2.32.3' \
    'pydantic>=2.11.7,<2.12.0' pydantic-settings==2.4.0 python-dotenv==1.0.1 \
    'supervision==0.27.0.post2' 'ultralytics>=8.4.37,<8.5.0' 'lap>=0.5.13,<0.6' \
    'transformers>=5.2.0,<5.3.0' 'pandas>=2.0' scipy
!pip install -q git+https://github.com/roboflow/sports.git@main
!pip install -q inference-gpu==1.3.0
!pip install -q onnxruntime-gpu==1.20.1 \
    --extra-index-url https://aiinfra.pkgs.visualstudio.com/PublicPackages/_packaging/onnxruntime-cuda-12/pypi/simple/

import onnxruntime as ort
print('onnxruntime providers:', ort.get_available_providers())
assert 'CUDAExecutionProvider' in ort.get_available_providers(), 'CUDA EP missing — re-run this cell.'
print('OK — GPU inference ready.')

In [ ]:
# Cell 3 — Clone or update the repo
import os, sys
BRANCH = 'claude/setup-gpu-video-testing-JhgUH'
REPO = 'https://github.com/muwafagq/playbook-program.git'
DEST = '/content/playbook'
if os.path.isdir(DEST + '/.git'):
    !git -C {DEST} fetch -q origin {BRANCH}
    !git -C {DEST} checkout -q {BRANCH}
    !git -C {DEST} reset -q --hard origin/{BRANCH}
else:
    !git clone -q --branch {BRANCH} {REPO} {DEST}
os.chdir(DEST); sys.path.insert(0, DEST)
head = !git -C {DEST} log --oneline -1
print('HEAD:', head[0])

In [ ]:
# Cell 4 — .env = baseline.env + API key (no source files are modified)
import shutil
shutil.copy(f'{DEST}/baseline.env', f'{DEST}/.env')
from google.colab import userdata
ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
with open(f'{DEST}/.env', 'a') as f:
    f.write(f'\nROBOFLOW_API_KEY={ROBOFLOW_API_KEY}\n')
print('.env configured.')

In [ ]:
# Cell 5 — Mount Drive, check inputs
import os
from google.colab import drive
drive.mount('/content/drive')

DATA = '/content/drive/MyDrive/Playbook/benchmark'
HALVES = {
    'half1': {'video': f'{DATA}/half1.mp4', 'gt': f'{DATA}/per_frame_tracks_half1_unified.csv', 'window': (1066, 3333)},
    'half2': {'video': f'{DATA}/half2.mp4', 'gt': f'{DATA}/per_frame_tracks_half2_unified.csv', 'window': (4270, 5720)},
}
WORK = '/content/bench'
os.makedirs(WORK, exist_ok=True)

missing = [p for h in HALVES.values() for p in (h['video'], h['gt']) if not os.path.exists(p)]
if missing:
    raise SystemExit('Missing in Drive:\n  ' + '\n  '.join(missing))

import cv2
for name, h in HALVES.items():
    cap = cv2.VideoCapture(h['video'])
    n, fps = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)), cap.get(cv2.CAP_PROP_FPS)
    w, ht = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    h['fps'] = fps
    ok = n > h['window'][1]
    print(f"{name}: {n} frames @ {fps:.2f} fps, {w}x{ht}  window {h['window']}  {'OK' if ok else 'VIDEO TOO SHORT — is this the full half?'}")
    if not ok:
        raise SystemExit(f'{name}: video ends before the window. Use the full, untrimmed half.')

In [ ]:
# Cell 6 — Run the pipeline unattended on each window (skips halves already done)
import subprocess, sys
env = os.environ.copy()
env.update({'DEVICE': 'cuda', 'ONNXRUNTIME_EXECUTION_PROVIDERS': 'CUDAExecutionProvider',
            'ROBOFLOW_API_KEY': ROBOFLOW_API_KEY, 'MAX_FRAMES': '0'})

for name, h in HALVES.items():
    out = f'{WORK}/{name}/run'
    if os.path.exists(f'{out}/kpi_summary.json'):
        print(f'{name}: already done, skipping.')
        continue
    os.makedirs(out, exist_ok=True)
    lo, hi = h['window']
    cmd = [sys.executable, f'{DEST}/main.py', '--source-video', h['video'], '--out-dir', out,
           '--enable-team', '--start-frame', str(lo), '--end-frame', str(hi)]
    print(f'{name}: running frames {lo}-{hi} ...', flush=True)
    with open(f'{out}/run.log', 'w') as log:
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
        for line in p.stdout:
            log.write(line)
            if line.startswith(('[stage]', '[WARN]', 'Done', 'Processed', 'Elapsed', 'Traceback')) or 'Error' in line:
                print('  ' + line.rstrip(), flush=True)
        p.wait()
    if p.returncode != 0:
        print(open(f'{out}/run.log').read()[-3000:])
        raise SystemExit(f'{name}: pipeline FAILED (code {p.returncode}). Full log: {out}/run.log')
print('All windows processed.')

In [ ]:
# Cell 7 — Score: proxy metrics (pred and truth) + accuracy against truth
import pandas as pd

def bench(*args):
    r = subprocess.run([sys.executable, '-m', 'tools.benchmark', *args], cwd=DEST,
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-2000:], r.stderr[-3000:])
        raise SystemExit('benchmark failed: ' + ' '.join(args[:1]))

for name, h in HALVES.items():
    lo, hi = h['window']
    base, fps = f'{WORK}/{name}', str(round(h['fps'], 3))
    gt = pd.read_csv(h['gt'], low_memory=False)
    gt[(gt.frame >= lo) & (gt.frame <= hi)].to_csv(f'{base}/gt_window.csv', index=False)
    bench('run', '--csv', f'{base}/run/per_frame_tracks.csv', '--kpi', f'{base}/run/kpi_summary.json',
          '--out-dir', f'{base}/bench_pred', '--fps', fps, '--spotcheck', '0')
    bench('run', '--csv', f'{base}/gt_window.csv', '--out-dir', f'{base}/bench_truth', '--fps', fps, '--spotcheck', '0')
    for id_col, tag in (('display_track_id', 'display'), ('track_id', 'stable')):
        bench('score-gt', '--pred', f'{base}/run/per_frame_tracks.csv', '--gt', f'{base}/gt_window.csv',
              '--pred-id-col', id_col, '--out-dir', f'{base}/score_{tag}', '--fps', fps)
    print(f'{name}: scored.')

In [ ]:
# Cell 8 — Summary
import json
def J(p): return json.load(open(p))
rows = []
for name in HALVES:
    b = f'{WORK}/{name}'
    s, st = J(f'{b}/score_display/score_gt.json'), J(f'{b}/score_stable/score_gt.json')
    bp, bt = J(f'{b}/bench_pred/benchmark.json'), J(f'{b}/bench_truth/benchmark.json')
    rows.append({
        'half': name,
        'minutes': round(bp['input']['duration_min'], 2),
        'IDF1 (shown ids)': s['IDF1'], 'IDF1 (stable ids)': st['IDF1'],
        'ID switches': s['id_switches'], 'switches / player-min': s['id_switches_per_player_minute'],
        'ids covering 2+ players': s['pred_ids_covering_2plus_players'],
        'real players': s['gt_players'], 'ids used': s['pred_ids'],
        'MOTA': s['MOTA'], 'det recall': s['detection_recall'], 'det precision': s['detection_precision'],
        'image jumps / player-min (pred)': bp['jumps']['image_jumps_per_player_minute'],
        'image jumps / player-min (truth)': bt['jumps']['image_jumps_per_player_minute'],
        'homography ok': bp['homography'].get('homography_ok_rate'),
        'warnings': '; '.join(s['warnings']) or '—',
    })
summary = pd.DataFrame(rows).set_index('half').T
summary.to_csv(f'{WORK}/summary.csv')
print(summary.to_string())
for name in HALVES:
    pp = pd.read_csv(f'{WORK}/{name}/score_display/gt_per_player.csv')
    print(f'\n{name} — worst-tracked players:')
    print(pp.head(6)[['gt_id', 'gt_frames', 'id_accuracy', 'id_switches', 'distinct_pred_ids']].to_string(index=False))

In [ ]:
# Cell 9 — Save results to Drive (videos left out of the zip to keep it small)
import time, shutil
dest = f"{DATA}/results_{time.strftime('%Y%m%d_%H%M')}"
shutil.copytree(WORK, dest, ignore=shutil.ignore_patterns('annotated.mp4', 'gt_window.csv'))
shutil.make_archive(dest, 'zip', dest)
for name in HALVES:
    src = f'{WORK}/{name}/run/annotated.mp4'
    if os.path.exists(src):
        shutil.copy(src, f'{dest}/{name}_annotated.mp4')
print('Saved to', dest, 'and', dest + '.zip')